# Advanced History Matching Configuration

This notebook demonstrates advanced configuration options and customization capabilities of the history matching library using the new object-oriented API.

## Overview

Building on the basic workflow, we'll explore:
- Advanced builder patterns and configuration options
- Custom sampling strategies and emulator types
- Feature selection strategies
- Interactive workflow control
- Performance tuning and diagnostics

We'll use the same SIR model but with more sophisticated configuration.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Import the history matching library
import history_matching as hm

%matplotlib inline
np.random.seed(42)

## SIR Model Setup (Reused from Basic Example)

We'll use the same stochastic SIR model for consistency.

In [ ]:
class SIR:
    """A stochastic SIR model using tau-leap algorithm."""
    
    def __init__(self, beta=1.2, gamma=0.3, s0=990, i0=10, r0=0, 
                 n_days=20, step=0.1, seed=None):
        # Validate inputs
        assert beta > 0 and gamma > 0, "Rates must be positive"
        assert s0 >= 0 and i0 >= 0 and r0 >= 0, "Initial conditions must be non-negative"
        assert n_days > 0 and 0 < step <= 1, "Invalid time parameters"
        
        self.beta = beta
        self.gamma = gamma
        self.s0 = int(s0)
        self.i0 = int(i0)
        self.r0 = int(r0)
        self.n_days = int(n_days)
        self.step = step
        self.seed = seed
        self.simulation_complete = False
    
    def run(self):
        """Run the stochastic SIR simulation."""
        # Setup
        n_steps = math.ceil((self.n_days - 1) / self.step)
        steps_per_day = int(1 / self.step)
        N = self.s0 + self.i0 + self.r0
        
        # Initialize arrays
        s = np.zeros(n_steps + 1, dtype=int)
        i = np.zeros(n_steps + 1, dtype=int)
        r = np.zeros(n_steps + 1, dtype=int)
        
        s[0] = self.s0
        i[0] = self.i0
        r[0] = self.r0
        
        # Daily outputs
        s_daily = np.zeros(self.n_days, dtype=int)
        i_daily = np.zeros(self.n_days, dtype=int)
        r_daily = np.zeros(self.n_days, dtype=int)
        
        s_daily[0] = s[0]
        i_daily[0] = i[0]
        r_daily[0] = r[0]
        
        if self.seed is not None:
            np.random.seed(self.seed)
        
        # Tau-leap simulation
        for j in range(1, n_steps + 1):
            # Calculate event rates
            lambda_transmission = (self.beta * s[j-1] * i[j-1] / N) * self.step
            lambda_recovery = self.gamma * i[j-1] * self.step
            
            # Sample events
            delta_Mt = np.random.poisson(lambda_transmission)
            delta_Mr = np.random.poisson(lambda_recovery)
            
            # Update states
            s[j] = max(s[j-1] - delta_Mt, 0)
            i[j] = max(i[j-1] + delta_Mt - delta_Mr, 0)
            r[j] = r[j-1] + delta_Mr
        
        # Extract daily values
        for j in range(1, self.n_days):
            k = j * steps_per_day
            s_daily[j] = s[k]
            i_daily[j] = i[k]
            r_daily[j] = r[k]
        
        self.s = s_daily
        self.i = i_daily
        self.r = r_daily
        self.simulation_complete = True
        
        return s_daily, i_daily, r_daily
    
    def get_incidence(self):
        """Calculate daily incidence (new infections)."""
        if not self.simulation_complete:
            self.run()
        
        incidence = np.zeros(len(self.i))
        incidence[0] = 0
        for j in range(1, len(self.i)):
            incidence[j] = self.s[j-1] - self.s[j]
        
        return incidence

# Generate synthetic data
beta_true = 1.3
gamma_true = 0.5
population_size = 10000
initial_infected = 100

true_model = SIR(
    beta=beta_true,
    gamma=gamma_true,
    s0=population_size - initial_infected,
    i0=initial_infected,
    seed=42
)

S_obs, I_obs, R_obs = true_model.run()
incidence_obs = true_model.get_incidence()

print(f"True parameters: β={beta_true}, γ={gamma_true}, R₀={beta_true/gamma_true:.2f}")
print(f"Peak incidence: {max(incidence_obs):.0f}, Total cases: {sum(incidence_obs):.0f}")

In [ ]:
def enhanced_sir_simulation(samples: pd.DataFrame) -> pd.DataFrame:
    """
    Enhanced simulation function that returns more features.
    """
    results = []
    
    for _, row in samples.iterrows():
        model = SIR(
            beta=row['beta'],
            gamma=row['gamma'],
            s0=population_size - initial_infected,
            i0=initial_infected
        )
        
        S, I, R = model.run()
        incidence = model.get_incidence()
        
        # Create comprehensive feature set
        result = {}
        
        # Time series features (every other day to reduce dimensionality)
        for i in range(0, len(incidence), 2):
            result[f'incidence_day_{i}'] = incidence[i]
            result[f'infected_day_{i}'] = I[i]
        
        # Summary statistics
        result['peak_incidence'] = max(incidence)
        result['peak_incidence_day'] = np.argmax(incidence)
        result['total_cases'] = sum(incidence)
        result['attack_rate'] = sum(incidence) / population_size
        result['final_size'] = R[-1]
        result['max_infected'] = max(I)
        
        # Epidemic characteristics
        result['time_to_peak'] = np.argmax(incidence)
        result['epidemic_duration'] = len([x for x in incidence[1:] if x > 0])
        result['growth_rate'] = np.log(incidence[3] / incidence[1]) if incidence[1] > 0 and incidence[3] > 0 else 0
        
        # Cumulative features
        cumulative_cases = np.cumsum(incidence)
        result['cases_week_1'] = cumulative_cases[6] if len(cumulative_cases) > 6 else cumulative_cases[-1]
        result['cases_week_2'] = cumulative_cases[13] if len(cumulative_cases) > 13 else cumulative_cases[-1]
        
        results.append(result)
    
    return pd.DataFrame(results)

# Test the enhanced simulation
test_samples = pd.DataFrame({
    'beta': [1.0, 1.5],
    'gamma': [0.3, 0.5]
})

test_results = enhanced_sir_simulation(test_samples)
print(f"Enhanced simulation produces {len(test_results.columns)} features:")
print(f"Features: {list(test_results.columns[:10])}...")

## Advanced Builder Configuration

The `HistoryMatchingBuilder` provides fine-grained control over all aspects of the history matching workflow.

In [ ]:
# Create parameter space using DataFrame for more control
parameter_df = pd.DataFrame({
    'parameter': ['beta', 'gamma'],
    'minimum': [0.5, 0.1],
    'maximum': [3.0, 1.0],
    'description': ['Transmission rate per day', 'Recovery rate per day']
})

# Create comprehensive observations - mix of summary stats and time series
observations_df = pd.DataFrame({
    'feature': [
        'peak_incidence',
        'total_cases', 
        'attack_rate',
        'time_to_peak',
        'incidence_day_4',
        'incidence_day_8',
        'incidence_day_12',
        'cases_week_1',
        'cases_week_2'
    ],
    'mean': [
        max(incidence_obs),
        sum(incidence_obs),
        sum(incidence_obs) / population_size,
        np.argmax(incidence_obs),
        incidence_obs[4],
        incidence_obs[8], 
        incidence_obs[12],
        sum(incidence_obs[:7]),
        sum(incidence_obs[:14])
    ],
    'variance': [
        (50)**2,    # Peak incidence uncertainty
        (200)**2,   # Total cases uncertainty
        (0.02)**2,  # Attack rate uncertainty
        (1)**2,     # Time to peak uncertainty (days)
        (30)**2,    # Daily incidence uncertainty
        (40)**2,
        (25)**2,
        (100)**2,   # Weekly totals uncertainty
        (150)**2
    ]
})

print(f"Advanced parameter space ({len(parameter_df)} parameters):")
print(parameter_df)
print(f"\nComprehensive observations ({len(observations_df)} features):")
print(observations_df)

In [ ]:
# Create advanced builder with full configuration control
print("🏗️  Building advanced history matching engine...")

builder = (hm.HistoryMatchingBuilder
    .from_dataframes(parameter_df, observations_df)
    .with_sampling_strategy({
        'type': 'lhs',
        'criterion': 'maximin',  # Maximize minimum distance between points
        'iterations': 10         # LHS optimization iterations
    })
    .with_feature_selection({
        'method': 'fano',           # Fano factor-based selection
        'max_features': 3,          # Limit features per iteration
        'correlation_threshold': 0.8 # Avoid highly correlated features
    })
    .with_emulator_type('gpr', 
        kernel='matern52',          # Matérn 5/2 kernel
        length_scale_bounds=(0.1, 10.0),
        noise_level=1e-5
    )
    .with_samples_per_iteration(800)
    .with_max_iterations(6)
    .with_implausibility_threshold(2.8)  # Slightly more conservative
    .with_space_reduction(False)         # Keep original space
    .with_oversample_factor(5.0)         # Higher oversampling for better filtering
    .with_random_seed(789)
)

# Preview the configuration before building
print("🔍 Configuration preview:")
config = builder.preview_configuration()
for key, value in config.items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for sub_key, sub_value in value.items():
            print(f"    {sub_key}: {sub_value}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Build the engine
advanced_engine = builder.build()
advanced_engine.set_simulation_function(enhanced_sir_simulation)

print(f"✅ Advanced engine built successfully!")
print(f"  Parameters: {len(advanced_engine.parameter_space.get_parameter_names())}")
print(f"  Observations: {len(observations_df)}")
print(f"  Sampling strategy: {advanced_engine._sampling_strategy.get_strategy_name()}")
print(f"  Feature selection: {advanced_engine._feature_selection_strategy.get_strategy_name()}")
print(f"  Emulator type: {type(advanced_engine._emulator_factory).__name__}")
print(f"  Implausibility threshold: {advanced_engine._implausibility_threshold}")
print(f"  Oversample factor: {advanced_engine._oversample_factor}")

## Interactive Workflow with Advanced Controls

Let's use the interactive workflow to demonstrate advanced features like strategy switching and diagnostics.

In [ ]:
# Add progress callback to monitor the workflow
def progress_callback(progress):
    print(f"  📊 Progress update: Iteration {progress.current_iteration}, "
          f"Acceptance rate: {progress.acceptance_rate:.3f}, "
          f"Total emulators: {progress.total_emulators_trained}")

advanced_engine.add_progress_callback(progress_callback)

print("🎮 Starting interactive advanced workflow...")
print(f"Initial parameter ranges:")
for param_name in advanced_engine.parameter_space.get_parameter_names():
    bounds = advanced_engine.parameter_space.get_bounds()[param_name]
    print(f"  {param_name}: [{bounds[0]}, {bounds[1]}]")

In [ ]:
# Iteration 1: Automatic feature selection
print("\n🔄 Iteration 1: Using automatic feature selection...")
result1 = advanced_engine.step()

print(f"✅ Iteration 1 results:")
print(f"  Samples generated: {len(result1.samples)}")
print(f"  Features selected: {result1.selected_features}")
print(f"  Parameter ranges after filtering:")
for param in ['beta', 'gamma']:
    values = result1.samples[param]
    print(f"    {param}: [{values.min():.3f}, {values.max():.3f}]")
    
# Check emulator quality (in real workflow, you'd inspect diagnostics)
print(f"  Emulators trained: {len(result1.emulators)}")

# Accept iteration 1
print("  ✅ Accepting iteration 1...")
advanced_engine.commit_step()

In [ ]:
# Iteration 2: Switch to manual feature selection
print("\n🔄 Iteration 2: Switching to manual feature selection...")

# Update strategy to focus on key summary statistics
advanced_engine.update_feature_selection(['peak_incidence', 'attack_rate'])
print(f"  Updated to manual feature selection: ['peak_incidence', 'attack_rate']")

result2 = advanced_engine.step()

print(f"✅ Iteration 2 results:")
print(f"  Samples generated: {len(result2.samples)}")
print(f"  Acceptance rate: {advanced_engine.acceptance_rate:.3f}")
print(f"  Features selected: {result2.selected_features}")
print(f"  Parameter ranges:")
for param in ['beta', 'gamma']:
    values = result2.samples[param]
    print(f"    {param}: [{values.min():.3f}, {values.max():.3f}]")

print("  ✅ Accepting iteration 2...")
advanced_engine.commit_step()

In [ ]:
# Iteration 3: Switch emulator type and continue
print("\n🔄 Iteration 3: Switching to linear emulator for comparison...")

# Switch to a simpler emulator to see the difference
advanced_engine.update_emulator_type('linear')
print(f"  Updated emulator type to: linear")

# Use time series features this time
advanced_engine.update_feature_selection(['incidence_day_4', 'incidence_day_8'])
print(f"  Updated to time series features: ['incidence_day_4', 'incidence_day_8']")

result3 = advanced_engine.step()

print(f"✅ Iteration 3 results:")
print(f"  Samples generated: {len(result3.samples)}")
print(f"  Acceptance rate: {advanced_engine.acceptance_rate:.3f}")
print(f"  Features selected: {result3.selected_features}")

# Check if linear emulator performed poorly
if advanced_engine.acceptance_rate < 0.1:
    print(f"  ⚠️ Low acceptance rate with linear emulator - reverting to GPR")
    advanced_engine.revert_step()  # Revert this iteration
    
    # Switch back to GPR and retry
    advanced_engine.update_emulator_type('gpr')
    result3 = advanced_engine.step()
    print(f"  ✅ Retried with GPR: {len(result3.samples)} samples, rate: {advanced_engine.acceptance_rate:.3f}")

print("  ✅ Accepting iteration 3...")
advanced_engine.commit_step()

print(f"\n🎯 Interactive workflow completed!")
print(f"  Total iterations: {advanced_engine.current_iteration}")
print(f"  Final acceptance rate: {advanced_engine.acceptance_rate:.3f}")

## Advanced Analysis and Diagnostics

In [ ]:
# Get all results for analysis
all_results = advanced_engine.get_all_results()
final_samples = all_results[-1].samples

print(f"📊 Advanced Analysis Results:")
print(f"  Total iterations completed: {len(all_results)}")
print(f"  Final plausible samples: {len(final_samples)}")
print(f"  Total emulators trained: {advanced_engine.progress.total_emulators_trained}")
print(f"  Total samples evaluated: {advanced_engine.progress.total_samples_generated}")

# Parameter estimation summary
print(f"\n🎯 Parameter Recovery Assessment:")
for param, true_value in [('beta', beta_true), ('gamma', gamma_true)]:
    values = final_samples[param]
    median_est = values.median()
    ci_lower = values.quantile(0.025)
    ci_upper = values.quantile(0.975)
    
    in_ci = ci_lower <= true_value <= ci_upper
    relative_error = abs(median_est - true_value) / true_value
    
    print(f"  {param}: Estimated {median_est:.3f} (95% CI: [{ci_lower:.3f}, {ci_upper:.3f}])")
    print(f"       True value: {true_value} {'✅' if in_ci else '❌'} ({'in' if in_ci else 'out of'} CI)")
    print(f"       Relative error: {relative_error:.1%}")

# Check R0 recovery
R0_samples = final_samples['beta'] / final_samples['gamma']
R0_true = beta_true / gamma_true
R0_median = R0_samples.median()
R0_ci_lower = R0_samples.quantile(0.025)
R0_ci_upper = R0_samples.quantile(0.975)
R0_in_ci = R0_ci_lower <= R0_true <= R0_ci_upper

print(f"\n📈 Derived Parameter (R₀):")
print(f"  Estimated: {R0_median:.3f} (95% CI: [{R0_ci_lower:.3f}, {R0_ci_upper:.3f}])")
print(f"  True value: {R0_true:.3f} {'✅' if R0_in_ci else '❌'}")

In [ ]:
# Advanced visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Parameter evolution across iterations
ax = axes[0, 0]
colors = ['lightblue', 'orange', 'lightgreen', 'red']
for i, result in enumerate(all_results):
    if i < len(colors):
        ax.scatter(result.samples['beta'], result.samples['gamma'], 
                  alpha=0.6, s=20, color=colors[i], label=f'Iteration {i+1}')

ax.axvline(beta_true, color='red', linestyle='--', linewidth=2, alpha=0.8)
ax.axhline(gamma_true, color='red', linestyle='--', linewidth=2, alpha=0.8)
ax.set_xlabel('β (transmission rate)')
ax.set_ylabel('γ (recovery rate)')
ax.set_title('Parameter Space Evolution')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Final parameter distributions
ax = axes[0, 1]
ax.hist(final_samples['beta'], bins=25, alpha=0.7, color='blue', density=True, label='β')
ax.axvline(beta_true, color='red', linestyle='--', linewidth=2, label=f'True β = {beta_true}')
ax.axvline(final_samples['beta'].median(), color='blue', linestyle='-', linewidth=2, 
           label=f'Est. β = {final_samples["beta"].median():.2f}')
ax.set_xlabel('β')
ax.set_ylabel('Density')
ax.set_title('β Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 2]
ax.hist(final_samples['gamma'], bins=25, alpha=0.7, color='orange', density=True, label='γ')
ax.axvline(gamma_true, color='red', linestyle='--', linewidth=2, label=f'True γ = {gamma_true}')
ax.axvline(final_samples['gamma'].median(), color='orange', linestyle='-', linewidth=2,
           label=f'Est. γ = {final_samples["gamma"].median():.2f}')
ax.set_xlabel('γ')
ax.set_ylabel('Density')
ax.set_title('γ Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: R0 distribution
ax = axes[1, 0]
ax.hist(R0_samples, bins=25, alpha=0.7, color='purple', density=True)
ax.axvline(R0_true, color='red', linestyle='--', linewidth=2, label=f'True R₀ = {R0_true:.2f}')
ax.axvline(R0_median, color='purple', linestyle='-', linewidth=2, label=f'Est. R₀ = {R0_median:.2f}')
ax.set_xlabel('R₀')
ax.set_ylabel('Density')
ax.set_title('R₀ Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Acceptance rate evolution
ax = axes[1, 1]
iterations = range(1, len(all_results) + 1)
sample_counts = [len(result.samples) for result in all_results]
ax.bar(iterations, sample_counts, alpha=0.7, color='green')
ax.set_xlabel('Iteration')
ax.set_ylabel('Plausible Samples')
ax.set_title('Sample Count per Iteration')
ax.grid(True, alpha=0.3)

# Plot 5: Feature selection summary
ax = axes[1, 2]
all_features = []
for result in all_results:
    all_features.extend(result.selected_features)

feature_counts = pd.Series(all_features).value_counts()
feature_counts.plot(kind='bar', ax=ax, color='teal', alpha=0.7)
ax.set_xlabel('Features')
ax.set_ylabel('Times Selected')
ax.set_title('Feature Selection Frequency')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📈 Feature Selection Summary:")
for feature, count in feature_counts.head().items():
    print(f"  {feature}: used {count} time(s)")

## Performance Comparison: Different Configurations

Let's compare different configuration strategies to understand their impact.

In [ ]:
def run_quick_comparison(config_name, **kwargs):
    """Run a quick 2-iteration comparison with different settings."""
    print(f"\n🔬 Testing {config_name}...")
    
    # Use simpler observations for speed
    simple_obs = {
        'peak_incidence': (max(incidence_obs), 50**2),
        'total_cases': (sum(incidence_obs), 200**2)
    }
    
    engine = hm.quick_setup(
        parameter_bounds={'beta': (0.5, 3.0), 'gamma': (0.1, 1.0)},
        observations=simple_obs,
        n_samples=200,  # Smaller for speed
        max_iterations=2,
        random_seed=999,
        **kwargs
    )
    
    engine.set_simulation_function(enhanced_sir_simulation)
    
    # Time the execution
    import time
    start_time = time.time()
    results = engine.run()
    end_time = time.time()
    
    final_samples = results[-1].samples
    
    # Calculate performance metrics
    beta_error = abs(final_samples['beta'].median() - beta_true) / beta_true
    gamma_error = abs(final_samples['gamma'].median() - gamma_true) / gamma_true
    
    print(f"  ⏱️ Runtime: {end_time - start_time:.1f}s")
    print(f"  📊 Final samples: {len(final_samples)}")
    print(f"  🎯 β error: {beta_error:.1%}, γ error: {gamma_error:.1%}")
    print(f"  📈 Final acceptance rate: {engine.acceptance_rate:.3f}")
    
    return {
        'config': config_name,
        'runtime': end_time - start_time,
        'samples': len(final_samples),
        'beta_error': beta_error,
        'gamma_error': gamma_error,
        'acceptance_rate': engine.acceptance_rate
    }

# Compare different configurations
comparisons = []

# Baseline: LHS + GPR
comparisons.append(run_quick_comparison(
    "LHS + GPR (baseline)",
    sampling_strategy='lhs',
    emulator_type='gpr'
))

# Linear emulator comparison
comparisons.append(run_quick_comparison(
    "LHS + Linear",
    sampling_strategy='lhs', 
    emulator_type='linear'
))

# Random sampling comparison
comparisons.append(run_quick_comparison(
    "Random + GPR",
    sampling_strategy='random',
    emulator_type='gpr'
))

# High threshold comparison
comparisons.append(run_quick_comparison(
    "LHS + GPR (conservative)",
    sampling_strategy='lhs',
    emulator_type='gpr',
    implausibility_threshold=2.5
))

In [ ]:
# Summarize comparison results
comparison_df = pd.DataFrame(comparisons)

print("\n📊 Configuration Comparison Summary:")
print("=" * 80)
print(f"{'Configuration':<25} {'Runtime(s)':<12} {'Samples':<10} {'β Error':<10} {'γ Error':<10} {'Acc. Rate':<10}")
print("=" * 80)

for _, row in comparison_df.iterrows():
    print(f"{row['config']:<25} {row['runtime']:<12.1f} {row['samples']:<10} "
          f"{row['beta_error']:<10.1%} {row['gamma_error']:<10.1%} {row['acceptance_rate']:<10.3f}")

print("\n🏆 Best performing configurations:")
print(f"  Fastest: {comparison_df.loc[comparison_df['runtime'].idxmin(), 'config']}")
print(f"  Most samples: {comparison_df.loc[comparison_df['samples'].idxmax(), 'config']}")
print(f"  Lowest β error: {comparison_df.loc[comparison_df['beta_error'].idxmin(), 'config']}")
print(f"  Lowest γ error: {comparison_df.loc[comparison_df['gamma_error'].idxmin(), 'config']}")

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Runtime comparison
ax = axes[0]
comparison_df.plot(x='config', y='runtime', kind='bar', ax=ax, color='skyblue')
ax.set_title('Runtime Comparison')
ax.set_ylabel('Time (seconds)')
ax.tick_params(axis='x', rotation=45)

# Error comparison
ax = axes[1]
x = range(len(comparison_df))
width = 0.35
ax.bar([i - width/2 for i in x], comparison_df['beta_error'], width, label='β error', alpha=0.7)
ax.bar([i + width/2 for i in x], comparison_df['gamma_error'], width, label='γ error', alpha=0.7)
ax.set_xlabel('Configuration')
ax.set_ylabel('Relative Error')
ax.set_title('Parameter Estimation Error')
ax.set_xticks(x)
ax.set_xticklabels([c[:15] + '...' if len(c) > 15 else c for c in comparison_df['config']], rotation=45)
ax.legend()

# Sample count vs acceptance rate
ax = axes[2]
scatter = ax.scatter(comparison_df['acceptance_rate'], comparison_df['samples'], 
                    s=100, alpha=0.7, c=comparison_df['runtime'], cmap='viridis')
ax.set_xlabel('Final Acceptance Rate')
ax.set_ylabel('Final Sample Count')
ax.set_title('Acceptance Rate vs Sample Count')
plt.colorbar(scatter, ax=ax, label='Runtime (s)')

# Add config labels
for i, row in comparison_df.iterrows():
    ax.annotate(row['config'][:10] + '...', 
               (row['acceptance_rate'], row['samples']),
               xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.tight_layout()
plt.show()

## Checkpoint and Resume Functionality

For long-running workflows, the engine supports saving and loading checkpoints.

In [ ]:
# Save current state as checkpoint
checkpoint_path = Path('./advanced_engine_checkpoint.pkl')
advanced_engine.save_checkpoint(checkpoint_path)
print(f"✅ Checkpoint saved to {checkpoint_path}")

# Demonstrate loading from checkpoint
print(f"\n🔄 Loading engine from checkpoint...")

# Create new strategies for loading (required parameters)
sampling_strategy = hm.SamplingStrategyFactory.create_strategy('lhs')
feature_strategy = hm.AutoFeatureSelection(method='fano', max_features=3)
emulator_factory = hm.EmulatorFactory('gpr')

loaded_engine = hm.HistoryMatchingEngine.load_checkpoint(
    checkpoint_path,
    sampling_strategy=sampling_strategy,
    feature_selection_strategy=feature_strategy, 
    emulator_factory=emulator_factory
)

loaded_engine.set_simulation_function(enhanced_sir_simulation)

print(f"✅ Engine loaded successfully!")
print(f"  Current iteration: {loaded_engine.current_iteration}")
print(f"  Total samples accepted: {loaded_engine.progress.total_samples_accepted}")
print(f"  Emulators trained: {loaded_engine.progress.total_emulators_trained}")

# Can continue from where we left off
if loaded_engine.current_iteration < loaded_engine._max_iterations:
    print(f"\n🚀 Continuing workflow from checkpoint...")
    
    # Run one more iteration to demonstrate
    next_result = loaded_engine.step()
    print(f"  ✅ Additional iteration completed: {len(next_result.samples)} samples")
    loaded_engine.commit_step()
    
    print(f"  📊 Updated progress: Iteration {loaded_engine.current_iteration}, "
          f"Rate: {loaded_engine.acceptance_rate:.3f}")

# Clean up checkpoint file
checkpoint_path.unlink()
print(f"\n🧹 Checkpoint file cleaned up")

## Summary

This notebook demonstrated advanced features of the history matching library:

### 🔧 **Advanced Configuration**
- **Builder Pattern**: Fine-grained control over all components
- **Custom Strategies**: Sampling, feature selection, and emulator configuration
- **DataFrame Inputs**: Structured parameter and observation definitions
- **Performance Tuning**: Threshold, oversampling, and optimization settings

### 🎮 **Interactive Workflow Control**
- **Step-by-Step Execution**: Manual control over each iteration
- **Strategy Switching**: Dynamic reconfiguration during workflow
- **Rollback Capability**: Revert unsatisfactory iterations
- **Progress Monitoring**: Real-time callbacks and diagnostics

### 📊 **Advanced Analysis**
- **Comprehensive Diagnostics**: Parameter recovery assessment
- **Performance Comparison**: Different configuration strategies
- **Evolution Tracking**: Parameter space reduction over iterations
- **Feature Selection Analysis**: Understanding which features matter most

### 💾 **Enterprise Features**
- **Checkpoint/Resume**: Save and restore workflow state
- **Extensible Design**: Custom emulators and strategies
- **Scalable Architecture**: Handle complex, high-dimensional problems

### 🎯 **Key Takeaways**

1. **Emulator Choice Matters**: GPR generally outperforms linear models for complex relationships
2. **Feature Selection Strategy**: Automatic selection often finds good features, manual gives control
3. **Sampling Strategy**: LHS typically provides better space coverage than random sampling
4. **Threshold Tuning**: Conservative thresholds (≤3.0) balance precision and coverage
5. **Interactive Workflow**: Powerful for research and model development scenarios

The object-oriented API provides the flexibility needed for advanced applications while maintaining ease of use for standard workflows.